# 🧬 Mô Hình Hợp Nhất Đa Phương Thức Dự Đoán Tương Tác Thuốc (Unified Multimodal DDI)
### Tích Hợp: Heterogeneous Graph (HGT) + DRKG Knowledge Graph + Chemical-Clinical Text (SMILES → Formula) + Vision Transformer (ViT)

---

### 📌 1. Mục Tiêu & Kiến Trúc Tổng Quan
Notebook này hiện thực hóa một hệ thống học sâu đa phương thức hiện đại bậc nhất cho bài toán dự đoán 65 phân lớp tương tác thuốc (Drug-Drug Interaction - DDI), kết hợp 4 nguồn tri thức toàn diện:

1. **Đồ Thị Dị Thể Cục Bộ (Local Heterogeneous Biological Graph):**
   - Xây dựng mạng dị thể 2,893 nút gồm: `Drug` (572), `Target` (1,162), `Enzyme` (202), `Pathway` (957).
   - Sử dụng **Heterogeneous Graph Transformer (HGT)** để truyền thông điệp tương hỗ đa tầng giữa các thực thể sinh học.
2. **Đồ Thị Tri Thức Y Sinh Toàn Cầu (Global DRKG TransE Embeddings):**
   - Thừa hưởng tri thức từ **5.87 triệu liên kết y sinh** của mạng lưới DRKG (Amazon AWS / DGL) qua vector nhúng tiền huấn luyện 400 chiều.
   - Kết hợp cùng HGT tạo thành cơ chế **Multi-Scale Knowledge Representation** (Tri thức đa tầng: Vi mô cục bộ + Vĩ mô toàn cầu).
3. **Văn Bản Y Sinh Tích Hợp Công Thức Hóa Học (Chemical & Clinical Text):**
   - Biên dịch chuỗi SMILES thành công thức phân tử chuẩn Hill ($\text{C}_9\text{H}_8\text{O}_4$, $\text{Cl}_2\text{H}_6\text{N}_2\text{Pt}$...) bằng RDKit.
   - Ghép nối công thức hóa học cùng 5 trường văn bản lâm sàng (`mechanism_of_action`, `indication`, `metabolism`...) đưa vào **Biomedical Text Transformer (PubMedBERT / MiniLM)**.
4. **Thị Giác Phân Tử 2D (Vision Transformer - ViT):**
   - Trích xuất đặc trưng thị giác từ ảnh cấu trúc 2D $224 \times 224$ RGB bằng backbone **ViT (Vision Transformer)** tiền huấn luyện.
5. **Đánh Giá Toàn Diện & Tracking W&B:**
   - Huấn luyện với **Class-Weighted CrossEntropy** xử lý mất cân bằng nhãn.
   - Tracking trực tiếp lên **Weights & Biases** với đầy đủ các chỉ số: Macro-F1, Macro-Precision, Macro-Recall, Micro-AUC, Micro-AUPR.
   - Đánh giá độc lập trên cả 3 kịch bản kiểm thử: $S_0$ (Transductive), $S_1$ (Semi-inductive), $S_2$ (Fully-inductive / Zero-shot).


In [2]:
!pip install -q rdkit torch_geometric timm sentence-transformers wandb


In [3]:
import os
import sys
import json
import time
import tarfile
import urllib.request
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Thư viện Hóa tin học & Học trên Đồ thị
import rdkit
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

import torch_geometric
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv

# Thư viện Thị giác & Ngôn ngữ
import timm
import torchvision.transforms as transforms
from sentence_transformers import SentenceTransformer

# Thư viện Đánh giá & Tracking
from sklearn.preprocessing import label_binarize
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, average_precision_score
import wandb

# Thiết lập Random Seed để tái lập kết quả khoa học
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Thiết bị tính toán: {device}")
if torch.cuda.is_available():
    print(f"✓ Tên GPU: {torch.cuda.get_device_name(0)}")


✓ Thiết bị tính toán: cuda
✓ Tên GPU: Tesla T4


## 1. Cấu Hình Đường Dẫn & Nạp Dữ Liệu
Tự động nhận diện môi trường làm việc (Kaggle Notebook hoặc Local Environment) và nạp đầy đủ các tập dữ liệu tương tác và đặc trưng thuốc.


In [4]:
# Cấu hình đường dẫn linh hoạt (Kaggle vs Local)
BASE_DIR = Path('/kaggle/input/datasets/tuantc2306/train-data')
DATA_DIR = Path('/kaggle/working/data')
WORKING_DIR = Path('/kaggle/working')


# Nạp bảng tương tác thuốc
train_csv_path = BASE_DIR/'train.csv'
val_csv_path = BASE_DIR/'val.csv'
test_s0_csv_path = BASE_DIR/'test_s0.csv'
test_s1_csv_path = BASE_DIR/'test_s1.csv'
test_s2_csv_path = BASE_DIR/'test_s2.csv'
drug_features_path = Path("/kaggle/input/datasets/tuantc2306/extracted-data/drug_features.json")


train_df = pd.read_csv(train_csv_path)
val_df = pd.read_csv(val_csv_path)
test_s0_df = pd.read_csv(test_s0_csv_path)
test_s1_df = pd.read_csv(test_s1_csv_path)
test_s2_df = pd.read_csv(test_s2_csv_path)

with open(drug_features_path, 'r', encoding='utf-8') as f:
    drug_features = json.load(f)

# Tạo danh sách cố định 572 thuốc và mapping ID sang Index (0..571)
all_drug_ids = sorted(list(drug_features.keys()))
drug2idx = {d_id: idx for idx, d_id in enumerate(all_drug_ids)}
idx2drug = {idx: d_id for d_id, idx in drug2idx.items()}

# Thư mục lưu cache đặc trưng ngoại tuyến
CACHE_DIR = WORKING_DIR / 'cache_features'
os.makedirs(CACHE_DIR, exist_ok=True)

# Thư mục ảnh phân tử 2D
IMAGES_DIR = Path("/kaggle/input/datasets/tuantc2306/image-data/drug_images")

print(f"• Số lượng thuốc tổng cộng: {len(all_drug_ids):,} thuốc")
print(f"• Cặp tương tác: Train={len(train_df):,}, Val={len(val_df):,}, S0={len(test_s0_df):,}, S1={len(test_s1_df):,}, S2={len(test_s2_df):,}")
print(f"• Thư mục lưu cache: {CACHE_DIR}")
print(f"• Thư mục ảnh phân tử: {IMAGES_DIR}")


• Số lượng thuốc tổng cộng: 572 thuốc
• Cặp tương tác: Train=17,276, Val=3,502, S0=3,502, S1=11,602, S2=1,382
• Thư mục lưu cache: /kaggle/working/cache_features
• Thư mục ảnh phân tử: /kaggle/input/datasets/tuantc2306/image-data/drug_images


## 2. Trích Xuất & Lưu Cache Đồ Thị Tri Thức Y Sinh Toàn Cầu (Global DRKG)
Tải tệp nén chính thức từ Amazon S3 (`drkg.tar.gz` ~206MB) và trích xuất vector nhúng tiền huấn luyện TransE 400 chiều cho 572 loại thuốc trong tập dữ liệu.


In [5]:
DRKG_CACHE_PATH = CACHE_DIR / 'drug_drkg_embeddings.pt'

if DRKG_CACHE_PATH.exists():
    print(f"✓ Đã tìm thấy tệp cache DRKG tại: {DRKG_CACHE_PATH}")
    drug_drkg_embeddings = torch.load(DRKG_CACHE_PATH, weights_only=False)
else:
    print("=== BẮT ĐẦU TẢI & TRÍCH XUẤT PRETRAINED DRKG EMBEDDINGS (206 MB) ===")
    drkg_tar_path = CACHE_DIR / 'drkg.tar.gz'
    drkg_url = "https://dgl-data.s3-us-west-2.amazonaws.com/dataset/DRKG/drkg.tar.gz"
    
    if not drkg_tar_path.exists():
        print(f"• Đang tải {drkg_url}...")
        urllib.request.urlretrieve(drkg_url, drkg_tar_path)
        print("✓ Tải hoàn tất!")

    # Giải nén 2 tệp cần thiết: entities.tsv và DRKG_TransE_l2_entity.npy
    print("• Đang giải nén các tệp embedding DRKG...")
    with tarfile.open(drkg_tar_path, 'r:gz') as tar:
        for member in tar.getmembers():
            if member.name.endswith('entities.tsv') or member.name.endswith('DRKG_TransE_l2_entity.npy'):
                member.name = os.path.basename(member.name)
                tar.extract(member, path=CACHE_DIR)

    # Nạp từ điển ánh xạ thực thể
    entities_file = CACHE_DIR / 'entities.tsv'
    entity_df = pd.read_csv(entities_file, sep='	', header=None, names=['entity_name', 'entity_id'])
    entity_map = dict(zip(entity_df['entity_name'], entity_df['entity_id']))
    
    # Nạp ma trận embedding NumPy
    emb_file = CACHE_DIR / 'DRKG_TransE_l2_entity.npy'
    drkg_matrix = np.load(emb_file)
    print(f"• Kích thước ma trận DRKG toàn cục: {drkg_matrix.shape} (97,238 thực thể, 400 chiều)")

    # Ánh xạ cho 572 thuốc của tập dữ liệu
    drkg_tensors = []
    found_count = 0
    all_known_embs = []

    for d_id in all_drug_ids:
        key = f"Compound::{d_id}"
        if key in entity_map:
            all_known_embs.append(drkg_matrix[entity_map[key]])

    mean_emb = np.mean(all_known_embs, axis=0) if all_known_embs else np.zeros(400)

    for d_id in all_drug_ids:
        key = f"Compound::{d_id}"
        if key in entity_map:
            drkg_tensors.append(drkg_matrix[entity_map[key]])
            found_count += 1
        else:
            drkg_tensors.append(mean_emb)

    drug_drkg_embeddings = torch.tensor(np.array(drkg_tensors), dtype=torch.float32)
    torch.save(drug_drkg_embeddings, DRKG_CACHE_PATH)
    print(f"✓ Ánh xạ thành công: {found_count} / {len(all_drug_ids)} thuốc có sẵn trong DRKG!")
    print(f"✓ Đã lưu cache DRKG tensor: {drug_drkg_embeddings.shape} tại {DRKG_CACHE_PATH}")

print(f"• Tensor DRKG shape: {drug_drkg_embeddings.shape} (572 thuốc, 400 chiều)")


=== BẮT ĐẦU TẢI & TRÍCH XUẤT PRETRAINED DRKG EMBEDDINGS (206 MB) ===
• Đang tải https://dgl-data.s3-us-west-2.amazonaws.com/dataset/DRKG/drkg.tar.gz...
✓ Tải hoàn tất!
• Đang giải nén các tệp embedding DRKG...


/tmp/ipykernel_49/1414840293.py:22: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=CACHE_DIR)


• Kích thước ma trận DRKG toàn cục: (97238, 400) (97,238 thực thể, 400 chiều)
✓ Ánh xạ thành công: 572 / 572 thuốc có sẵn trong DRKG!
✓ Đã lưu cache DRKG tensor: torch.Size([572, 400]) tại /kaggle/working/cache_features/drug_drkg_embeddings.pt
• Tensor DRKG shape: torch.Size([572, 400]) (572 thuốc, 400 chiều)


## 3. Chuyển Đổi SMILES Thành Công Thức Hóa Học & Cache Text Embeddings
Chuyển đổi chuỗi SMILES thành công thức phân tử chuẩn Hill ($\text{C}_9\text{H}_8\text{O}_4$, $\text{Cl}_2\text{H}_6\text{N}_2\text{Pt}$...) bằng RDKit, cấu trúc prompt văn bản đa tầng và trích xuất vector ngữ nghĩa bằng mô hình ngôn ngữ y sinh (**PubMedBERT / MiniLM**).


In [6]:
TEXT_CACHE_PATH = CACHE_DIR / 'drug_text_embeddings.pt'

SUPPLEMENTARY_SMILES = {
    'DB01169': 'O1[As]2O[As]3O[As]1O[As](O2)O3',                                                         # Arsenic trioxide
        'DB00515': '[H][N]([H])([H])[Pt](Cl)(Cl)[N]([H])([H])[H]',                                                        # Cisplatin
        'DB13955': 'CCCCCCC(=O)O[C@H]1CC[C@H]2[C@@H]3CCC4=CC(OC(=O)CCCCCC)=CC=C4[C@H]3CC[C@]12C', # Estradiol dienanthate
        'DB09338': 'COC(CNC(=O)C1=CC=CC=C1OCC(O)=O)C[Hg]O',       # Mersalyl
        'DB00006': 'CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@H](CCC(O)=O)NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)[C@H](CC(O)=O)NC(=O)CNC(=O)[C@H](CC(N)=O)NC(=O)CNC(=O)CNC(=O)CNC(=O)CNC(=O)[C@@H]1CCCN1C(=O)[C@H](CCCNC(N)=N)NC(=O)[C@@H]1CCCN1C(=O)[C@H](N)CC1=CC=CC=C1)C(=O)N1CCC[C@H]1C(=O)N[C@@H](CCC(O)=O)C(=O)N[C@@H](CCC(O)=O)C(=O)N[C@@H](CC1=CC=C(O)C=C1)C(=O)N[C@@H](CC(C)C)C(O)=O' # Bivalirudin
}

def get_clean_smiles(d_id, feat):
    smi = feat.get('smiles')
    if smi and isinstance(smi, str) and smi.strip():
        return smi.strip()
    return SUPPLEMENTARY_SMILES.get(d_id, "")

def smiles_to_chemical_formula(smiles):
    """Biến đổi chuỗi SMILES sang công thức phân tử chuẩn hóa học (Hill System)"""
    if not smiles:
        return "N/A"
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            return rdMolDescriptors.CalcMolFormula(mol)
    except Exception:
        pass
    return "N/A"

if TEXT_CACHE_PATH.exists():
    print(f"✓ Đã tìm thấy tệp cache Text Embeddings tại: {TEXT_CACHE_PATH}")
    drug_text_embeddings = torch.load(TEXT_CACHE_PATH, weights_only=False)
else:
    print("=== BẮT ĐẦU TẠO PROMPT VĂN BẢN HÓA HỌC & TRÍCH XUẤT TEXT EMBEDDINGS ===")
    t_start = time.time()
    
    text_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
    
    rich_text_list = []
    formula_samples = []
    
    for d_id in all_drug_ids:
        feat = drug_features[d_id]
        name = feat.get('name', 'Unknown')
        smiles = get_clean_smiles(d_id, feat)
        formula = smiles_to_chemical_formula(smiles)
        
        if len(formula_samples) < 5:
            formula_samples.append((name, formula, smiles[:20] + '..'))
            
        desc = feat.get('description', '') or ''
        ind = feat.get('indication', '') or ''
        moa = feat.get('mechanism_of_action', '') or ''
        pd_act = feat.get('pharmacodynamics', '') or ''
        metab = feat.get('metabolism', '') or ''
        
        # Cấu trúc prompt đa tầng tích hợp Công thức Hóa học + SMILES + Văn bản Lâm sàng
        prompt = (
            f"Drug: {name}. Chemical Formula: {formula}. Molecular Structure SMILES: {smiles}. "
            f"Mechanism of Action: {moa[:250]}. "
            f"Indication: {ind[:200]}. "
            f"Pharmacodynamics: {pd_act[:150]}. "
            f"Metabolism: {metab[:150]}."
        )
        rich_text_list.append(prompt)

    print("• Mẫu một số công thức hóa học vừa chuyển đổi từ SMILES:")
    for n, f, s in formula_samples:
        print(f"  - {n:15s} | Formula: {f:12s} | SMILES: {s}")

    print(f"• Đang trích xuất embeddings cho {len(rich_text_list)} văn bản thuốc...")
    embs = text_model.encode(rich_text_list, batch_size=32, show_progress_bar=True, convert_to_numpy=True)
    drug_text_embeddings = torch.tensor(embs, dtype=torch.float32)
    torch.save(drug_text_embeddings, TEXT_CACHE_PATH)
    print(f"✓ Hoàn thành trong {time.time() - t_start:.2f}s! Đã lưu tại: {TEXT_CACHE_PATH}")

print(f"• Tensor Text shape: {drug_text_embeddings.shape} (572 thuốc, 384 chiều)")


=== BẮT ĐẦU TẠO PROMPT VĂN BẢN HÓA HỌC & TRÍCH XUẤT TEXT EMBEDDINGS ===


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (7 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

• Mẫu một số công thức hóa học vừa chuyển đổi từ SMILES:
  - Bivalirudin     | Formula: C98H138N24O33 | SMILES: CC[C@H](C)[C@H](NC(=..
  - Desmopressin    | Formula: C46H64N14O12S2 | SMILES: N=C(N)NCCC[C@@H](NC(..
  - Cyclosporine    | Formula: C62H111N11O12 | SMILES: C/C=C/C[C@@H](C)[C@@..
  - Cyanocobalamin  | Formula: N/A          | SMILES: ..
  - Choline         | Formula: C5H14NO+     | SMILES: C[N+](C)(C)CCO..
• Đang trích xuất embeddings cho 572 văn bản thuốc...


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

✓ Hoàn thành trong 11.53s! Đã lưu tại: /kaggle/working/cache_features/drug_text_embeddings.pt
• Tensor Text shape: torch.Size([572, 384]) (572 thuốc, 384 chiều)


## 4. Trích Xuất & Lưu Cache Thị Giác Phân Tử 2D (Vision Transformer - ViT)
Trích xuất đặc trưng thị giác từ ảnh cấu trúc phân tử $224 \times 224$ RGB bằng backbone **Vision Transformer (ViT)** tiền huấn luyện.


In [7]:
VIT_CACHE_PATH = CACHE_DIR / 'drug_vit_embeddings.pt'

if VIT_CACHE_PATH.exists():
    print(f"✓ Đã tìm thấy tệp cache ViT Embeddings tại: {VIT_CACHE_PATH}")
    drug_vit_embeddings = torch.load(VIT_CACHE_PATH, weights_only=False)
else:
    print("=== BẮT ĐẦU TRÍCH XUẤT ĐẶC TRƯNG ẢNH BẰNG VISION TRANSFORMER (ViT) ===")
    t_start = time.time()
    
    vit_model = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=0).to(device)
    vit_model.eval()
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    vit_feature_list = []
    
    default_img = Image.new('RGB', (224, 224), color=(240, 240, 240))
    default_tensor = transform(default_img).unsqueeze(0).to(device)
    with torch.no_grad():
        default_feat = vit_model(default_tensor).squeeze(0).cpu()

    for idx, d_id in enumerate(all_drug_ids):
        img_path = IMAGES_DIR / f"{d_id}.png" if IMAGES_DIR else None
        if img_path and img_path.exists():
            try:
                img = Image.open(img_path).convert('RGB')
                t_img = transform(img).unsqueeze(0).to(device)
                with torch.no_grad():
                    feat = vit_model(t_img).squeeze(0).cpu()
                vit_feature_list.append(feat)
            except Exception:
                vit_feature_list.append(default_feat)
        else:
            vit_feature_list.append(default_feat)

    drug_vit_embeddings = torch.stack(vit_feature_list)
    torch.save(drug_vit_embeddings, VIT_CACHE_PATH)
    print(f"✓ Hoàn thành trong {time.time() - t_start:.2f}s! Đã lưu tại: {VIT_CACHE_PATH}")

print(f"• Tensor ViT shape: {drug_vit_embeddings.shape} (572 thuốc, {drug_vit_embeddings.shape[1]} chiều)")


=== BẮT ĐẦU TRÍCH XUẤT ĐẶC TRƯNG ẢNH BẰNG VISION TRANSFORMER (ViT) ===


model.safetensors: reconstructing file:   0%|          |  0.00B / 22.9MB            

model.safetensors: downloading bytes:           |  0.00B            

✓ Hoàn thành trong 25.36s! Đã lưu tại: /kaggle/working/cache_features/drug_vit_embeddings.pt
• Tensor ViT shape: torch.Size([572, 192]) (572 thuốc, 192 chiều)


## 5. Xây Dựng Đồ Thị Dị Thể Sinh Học Cục Bộ (PyG `HeteroData`)
Tạo mạng dị thể gồm 4 loại thực thể: `drug` (572), `target` (1,162), `enzyme` (202), `pathway` (957) và 6 loại cạnh hai chiều (`targets`, `metabolized_by`, `participates_in`).


In [8]:
HETERO_GRAPH_CACHE_PATH = CACHE_DIR / 'biological_hetero_graph.pt'

target_vocab = sorted(list(set(t for feat in drug_features.values() for t in (feat.get('target') or []))))
enzyme_vocab = sorted(list(set(e for feat in drug_features.values() for e in (feat.get('enzyme') or []))))
pathway_vocab = sorted(list(set(p for feat in drug_features.values() for p in (feat.get('pathway') or []))))

target2idx = {t: i for i, t in enumerate(target_vocab)}
enzyme2idx = {e: i for i, e in enumerate(enzyme_vocab)}
pathway2idx = {p: i for i, p in enumerate(pathway_vocab)}

print(f"• Thống kê các loại nút sinh học:")
print(f"  - Drug:    {len(all_drug_ids):,} nút")
print(f"  - Target:  {len(target_vocab):,} nút (Protein Receptors / Transporters)")
print(f"  - Enzyme:  {len(enzyme_vocab):,} nút (CYP450 & Metabolic Enzymes)")
print(f"  - Pathway: {len(pathway_vocab):,} nút (KEGG Biological Pathways)")
print(f"  - TỔNG CỘNG: {len(all_drug_ids) + len(target_vocab) + len(enzyme_vocab) + len(pathway_vocab):,} nút")

drug_target_src, drug_target_dst = [], []
drug_enzyme_src, drug_enzyme_dst = [], []
drug_pathway_src, drug_pathway_dst = [], []

for d_id, feat in drug_features.items():
    u_idx = drug2idx[d_id]
    for t in (feat.get('target') or []):
        if t in target2idx:
            drug_target_src.append(u_idx)
            drug_target_dst.append(target2idx[t])
    for e in (feat.get('enzyme') or []):
        if e in enzyme2idx:
            drug_enzyme_src.append(u_idx)
            drug_enzyme_dst.append(enzyme2idx[e])
    for p in (feat.get('pathway') or []):
        if p in pathway2idx:
            drug_pathway_src.append(u_idx)
            drug_pathway_dst.append(pathway2idx[p])

hetero_data = HeteroData()

hetero_data['drug'].num_nodes = len(all_drug_ids)
hetero_data['target'].num_nodes = len(target_vocab)
hetero_data['enzyme'].num_nodes = len(enzyme_vocab)
hetero_data['pathway'].num_nodes = len(pathway_vocab)

feat_dim = 128
torch.manual_seed(42)
hetero_data['drug'].x = torch.randn(len(all_drug_ids), feat_dim)
hetero_data['target'].x = torch.randn(len(target_vocab), feat_dim)
hetero_data['enzyme'].x = torch.randn(len(enzyme_vocab), feat_dim)
hetero_data['pathway'].x = torch.randn(len(pathway_vocab), feat_dim)

dt_edge = torch.tensor([drug_target_src, drug_target_dst], dtype=torch.long)
hetero_data['drug', 'targets', 'target'].edge_index = dt_edge
hetero_data['target', 'targeted_by', 'drug'].edge_index = dt_edge.flip(0)

de_edge = torch.tensor([drug_enzyme_src, drug_enzyme_dst], dtype=torch.long)
hetero_data['drug', 'metabolized_by', 'enzyme'].edge_index = de_edge
hetero_data['enzyme', 'metabolizes', 'drug'].edge_index = de_edge.flip(0)

dp_edge = torch.tensor([drug_pathway_src, drug_pathway_dst], dtype=torch.long)
hetero_data['drug', 'participates_in', 'pathway'].edge_index = dp_edge
hetero_data['pathway', 'participated_by', 'drug'].edge_index = dp_edge.flip(0)

hetero_data = hetero_data.to(device)
print("✓ Cấu trúc Đồ Thị Dị Thể PyG HeteroData:")
print(hetero_data)


• Thống kê các loại nút sinh học:
  - Drug:    572 nút
  - Target:  1,162 nút (Protein Receptors / Transporters)
  - Enzyme:  202 nút (CYP450 & Metabolic Enzymes)
  - Pathway: 957 nút (KEGG Biological Pathways)
  - TỔNG CỘNG: 2,893 nút
✓ Cấu trúc Đồ Thị Dị Thể PyG HeteroData:
HeteroData(
  drug={
    num_nodes=572,
    x=[572, 128],
  },
  target={
    num_nodes=1162,
    x=[1162, 128],
  },
  enzyme={
    num_nodes=202,
    x=[202, 128],
  },
  pathway={
    num_nodes=957,
    x=[957, 128],
  },
  (drug, targets, target)={ edge_index=[2, 3047] },
  (target, targeted_by, drug)={ edge_index=[2, 3047] },
  (drug, metabolized_by, enzyme)={ edge_index=[2, 2133] },
  (enzyme, metabolizes, drug)={ edge_index=[2, 2133] },
  (drug, participates_in, pathway)={ edge_index=[2, 2778] },
  (pathway, participated_by, drug)={ edge_index=[2, 2778] }
)


## 6. PyTorch Dataset & DataLoader
Xây dựng lớp Dataset nạp các chỉ số cặp thuốc `(idx1, idx2)` và nhãn tương tác phân loại 65 lớp.


In [9]:
class UnifiedMultimodalDataset(Dataset):
    """Dataset nạp cặp thuốc và nhãn tương tác"""
    def __init__(self, df, drug2idx):
        self.df = df.reset_index(drop=True)
        self.d1_indices = [drug2idx[d1] for d1 in self.df['drug1_id']]
        self.d2_indices = [drug2idx[d2] for d2 in self.df['drug2_id']]
        self.labels = self.df['label'].values if 'label' in self.df.columns else None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        idx1 = self.d1_indices[idx]
        idx2 = self.d2_indices[idx]
        if self.labels is not None:
            return idx1, idx2, torch.tensor(self.labels[idx], dtype=torch.long)
        return idx1, idx2

batch_size = 256

train_dataset = UnifiedMultimodalDataset(train_df, drug2idx)
val_dataset = UnifiedMultimodalDataset(val_df, drug2idx)
test_s0_dataset = UnifiedMultimodalDataset(test_s0_df, drug2idx)
test_s1_dataset = UnifiedMultimodalDataset(test_s1_df, drug2idx)
test_s2_dataset = UnifiedMultimodalDataset(test_s2_df, drug2idx)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_s0_loader = DataLoader(test_s0_dataset, batch_size=batch_size, shuffle=False)
test_s1_loader = DataLoader(test_s1_dataset, batch_size=batch_size, shuffle=False)
test_s2_loader = DataLoader(test_s2_dataset, batch_size=batch_size, shuffle=False)

print(f"✓ DataLoader batches: Train={len(train_loader)}, Val={len(val_loader)}, S0={len(test_s0_loader)}, S1={len(test_s1_loader)}, S2={len(test_s2_loader)}")


✓ DataLoader batches: Train=68, Val=14, S0=14, S1=46, S2=6


## 7. Kiến Trúc Mô Hình Hợp Nhất Đa Phương Thức (`UnifiedMultimodalDDIModel`)
Mô hình kết hợp:
1. **Biological HGT Encoder:** `HGTConv` 2 tầng trên đồ thị dị thể sinh học.
2. **DRKG Knowledge Projector:** Chiếu vector 400D từ DRKG.
3. **Multi-Scale Knowledge Fusion:** Dung hợp tri thức cục bộ (HGT) và toàn cầu (DRKG).
4. **Chemical-Clinical Text Projector:** Chiếu vector 384D từ PubMedBERT / MiniLM.
5. **ViT Vision Projector:** Chiếu vector 192D từ Vision Transformer.
6. **Cross-Modal Attention / Gated Fusion:** Dung hợp tất cả phương thức thành vector biểu diễn thuốc $h_{drug} \in \mathbb{R}^{384}$.
7. **Symmetric Pair Interaction & Classifier Head:** Dự đoán phân loại 65 lớp DDI.


In [15]:
class BiologicalHGTEncoder(nn.Module):
    """Mạng Heterogeneous Graph Transformer (HGT) truyền thông điệp qua Target, Enzyme, Pathway"""
    def __init__(self, metadata, hidden_dim=128, out_dim=256, num_heads=4, num_layers=2):
        super().__init__()
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            conv = HGTConv(
                in_channels=hidden_dim,
                out_channels=hidden_dim,
                metadata=metadata,
                heads=num_heads
            )
            self.convs.append(conv)
            
        self.drug_projector = nn.Sequential(
            nn.Linear(hidden_dim, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU()
        )

    def forward(self, x_dict, edge_index_dict):
        curr_x_dict = x_dict
        for conv in self.convs:
            curr_x_dict = conv(curr_x_dict, edge_index_dict)
            curr_x_dict = {key: F.relu(x) for key, x in curr_x_dict.items()}
        drug_emb = self.drug_projector(curr_x_dict['drug'])
        return drug_emb


class UnifiedMultimodalDDIModel(nn.Module):
    """Mô hình Hợp nhất Đa phương thức: HGT + DRKG + Text + ViT"""
    def __init__(
        self,
        hetero_metadata,
        drkg_dim=400,
        text_dim=384,
        vit_dim=192,
        proj_dim=256,
        fused_dim=384,
        num_classes=65
    ):
        super().__init__()
        
        # 1. Nhánh Đồ thị Dị thể Cục bộ (HGT)
        self.hgt_encoder = BiologicalHGTEncoder(hetero_metadata, hidden_dim=128, out_dim=proj_dim, num_heads=4, num_layers=2)
        
        # 2. Nhánh Đồ thị Tri thức Toàn cầu (DRKG)
        self.drkg_projector = nn.Sequential(
            nn.Linear(drkg_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(proj_dim, proj_dim)
        )
        
        # Cổng dung hợp Tri thức Đa tầng (HGT + DRKG)
        self.gate_knowledge = nn.Sequential(
            nn.Linear(proj_dim * 2, proj_dim),
            nn.Sigmoid()
        )
        
        # 3. Nhánh Văn bản Hóa học & Lâm sàng (Text)
        self.text_projector = nn.Sequential(
            nn.Linear(text_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
        # 4. Nhánh Thị giác Phân tử 2D (ViT)
        self.vit_projector = nn.Sequential(
            nn.Linear(vit_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
        # 5. Cổng dung hợp Đa phương thức (Knowledge + Text + ViT)
        self.modal_gate = nn.Sequential(
            nn.Linear(proj_dim * 3, 3),
            nn.Softmax(dim=-1)
        )
        self.fusion_out = nn.Sequential(
            nn.Linear(proj_dim, fused_dim),
            nn.LayerNorm(fused_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
        # 6. Bộ phân loại Tương tác Đối xứng (Symmetric Classifier)
        pair_dim = fused_dim * 3
        self.classifier = nn.Sequential(
            nn.Linear(pair_dim, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def compute_drug_representations(self, hetero_data, drkg_embs, text_embs, vit_embs):
        """Tính toán vector đại diện đa phương thức cho toàn bộ 572 loại thuốc"""
        # HGT forward
        h_hgt = self.hgt_encoder(hetero_data.x_dict, hetero_data.edge_index_dict)
        
        # DRKG forward
        h_drkg = self.drkg_projector(drkg_embs)
        
        # Dung hợp Tri thức (Multi-scale Knowledge)
        g_k = self.gate_knowledge(torch.cat([h_hgt, h_drkg], dim=-1))
        h_knowledge = g_k * h_hgt + (1.0 - g_k) * h_drkg
        
        # Text forward
        h_text = self.text_projector(text_embs)
        
        # ViT forward
        h_vit = self.vit_projector(vit_embs)
        
        # Dung hợp Đa phương thức có trọng số chú ý (Modality Gating)
        stacked_mods = torch.stack([h_knowledge, h_text, h_vit], dim=1) # [572, 3, proj_dim]
        gate_weights = self.modal_gate(torch.cat([h_knowledge, h_text, h_vit], dim=-1)).unsqueeze(-1) # [572, 3, 1]
        
        h_fused = torch.sum(stacked_mods * gate_weights, dim=1) # [572, proj_dim]
        h_drugs = self.fusion_out(h_fused) # [572, fused_dim]
        return h_drugs

    def forward(self, idx1, idx2, all_drug_reps=None, hetero_data=None, drkg_embs=None, text_embs=None, vit_embs=None):
        """Dự đoán nhãn tương tác cho mini-batch cặp thuốc"""
        if all_drug_reps is None:
            all_drug_reps = self.compute_drug_representations(hetero_data, drkg_embs, text_embs, vit_embs)
            
        h1 = all_drug_reps[idx1]
        h2 = all_drug_reps[idx2]
        
        # Tương tác đối xứng (Symmetric Interaction)
        feat_sum = h1 + h2
        feat_diff = torch.abs(h1 - h2)
        feat_prod = h1 * h2
        
        pair_feat = torch.cat([feat_sum, feat_diff, feat_prod], dim=-1)
        logits = self.classifier(pair_feat)
        return logits


## 8. Hàm Mất Mát & Đánh Giá Toàn Diện (Macro F1, Macro Precision, Macro Recall, Micro AUC, Micro AUPR)
Tính toán trọng số nghịch đảo tần suất lớp để xử lý mất cân bằng nhãn và hàm đánh giá đầy đủ các chỉ số: Accuracy, Macro-F1, Macro-Precision, Macro-Recall, Micro-AUC và Micro-AUPR theo chuẩn khoa học.


In [16]:
num_classes = 65
label_counts = Counter(train_df['label'].values)
total_samples = len(train_df)

class_weights = []
for c in range(num_classes):
    cnt = label_counts.get(c, 1)
    w = total_samples / (num_classes * cnt)
    class_weights.append(min(w, 25.0))

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
print(f"✓ Đã khởi tạo Class-Weighted CrossEntropyLoss (Min weight: {min(class_weights):.2f}, Max: {max(class_weights):.2f})")

def evaluate_model(model, loader, hetero_data, drkg_embs, text_embs, vit_embs, device, num_classes=65):
    model.eval()
    all_preds = []
    all_targets = []
    all_probs = []
    
    with torch.no_grad():
        all_drug_reps = model.compute_drug_representations(hetero_data, drkg_embs, text_embs, vit_embs)
        
        for idx1, idx2, labels in loader:
            idx1, idx2 = idx1.to(device), idx2.to(device)
            logits = model(idx1, idx2, all_drug_reps)
            probs = F.softmax(logits, dim=-1).cpu().numpy()
            preds = np.argmax(probs, axis=-1)
            
            all_preds.extend(preds)
            all_targets.extend(labels.numpy())
            all_probs.extend(probs)
            
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    all_probs = np.array(all_probs)
    
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    macro_precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    macro_recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    
    # Binarize nhãn sang ma trận [N, num_classes] để tính toán chuẩn xác Micro-AUC và Micro-AUPR toàn cục
    y_true_bin = label_binarize(all_targets, classes=list(range(num_classes)))
    
    # 1. Micro-AUC
    try:
        micro_auc = float(roc_auc_score(y_true_bin, all_probs, average='micro'))
    except Exception:
        try:
            micro_auc = float(roc_auc_score(y_true_bin.ravel(), all_probs.ravel()))
        except Exception:
            micro_auc = 0.0

    # 2. Micro-AUPR
    try:
        micro_aupr = float(average_precision_score(y_true_bin, all_probs, average='micro'))
    except Exception:
        try:
            micro_aupr = float(average_precision_score(y_true_bin.ravel(), all_probs.ravel()))
        except Exception:
            micro_aupr = 0.0
    
    return {
        'acc': acc,
        'macro_f1': macro_f1,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'micro_auc': micro_auc,
        'micro_aupr': micro_aupr
    }

print("✓ Đã định nghĩa hàm evaluate_model thành công!")


✓ Đã khởi tạo Class-Weighted CrossEntropyLoss (Min weight: 0.06, Max: 25.00)
✓ Đã định nghĩa hàm evaluate_model thành công!


## 9. Vòng Lặp Huấn Luyện Đa Learning Rate (Tối Đa 100 Epochs) & Tracking Weights & Biases
Thực nghiệm trên nhiều mức Learning Rate khác nhau với tối đa 100 epochs, kết hợp `ReduceLROnPlateau`, `Early Stopping`, đồng bộ hóa từng run lên WandB Dashboard và tự động chọn ra checkpoint tối ưu nhất toàn cục.


In [12]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("entrance_testing")

In [13]:
wandb.login(key=secret_value_0)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: noct2306 (noct2306-uet) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [20]:
# Chuyển các tensor đặc trưng lên GPU một lần duy nhất
drkg_embs_dev = drug_drkg_embeddings.to(device)
text_embs_dev = drug_text_embeddings.to(device)
vit_embs_dev = drug_vit_embeddings.to(device)

# ==============================================================================
# CẤU HÌNH THỰC NGHIỆM ĐA LEARNING RATE VỚI TỐI ĐA 100 EPOCHS
# ==============================================================================
LEARNING_RATES = [1e-3, 5e-4, 1e-4]  # Danh sách Learning Rate muốn thử nghiệm (có thể thêm/bớt tùy ý)
MAX_EPOCHS = 100                      # Huấn luyện tối đa 100 epochs
PATIENCE = 12                         # Dừng sớm nếu không cải thiện sau 12 epochs

WANDB_PROJECT = "DDI_Entrance_Test"
best_checkpoint_path = WORKING_DIR / 'best_unified_multimodal_model.pt'

global_best_val_f1 = -1.0
global_best_epoch = -1
global_best_lr = None
lr_experiment_records = []

print("=" * 80)
print(f"🚀 BẮT ĐẦU THỰC NGHIỆM {len(LEARNING_RATES)} MỨC LEARNING RATE VỚI TỐI ĐA {MAX_EPOCHS} EPOCHS")
print(f"• Danh sách Learning Rate: {LEARNING_RATES}")
print(f"• Early Stopping Patience: {PATIENCE} epochs")
print("=" * 80)

for lr_idx, lr in enumerate(LEARNING_RATES, 1):
    print(f"\n" + "-" * 75)
    print(f"▶ [{lr_idx}/{len(LEARNING_RATES)}] THỬ NGHIỆM VỚI LEARNING RATE: {lr:.0e} (Tối đa {MAX_EPOCHS} epochs)")
    print("-" * 75)
    
    # 1. Khởi tạo lại Random Seed để đảm bảo điều kiện xuất phát đồng đều
    set_seed(42)
    
    # 2. Khởi tạo mô hình mới cho mức LR này
    model = UnifiedMultimodalDDIModel(
        hetero_metadata=hetero_data.metadata(),
        drkg_dim=400,
        text_dim=384,
        vit_dim=drug_vit_embeddings.shape[1],
        proj_dim=256,
        fused_dim=384,
        num_classes=num_classes
    ).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    
    # 3. Khởi tạo WandB Run riêng cho mức LR này
    run_name = f"hgt-drkg-text-vit-lr_{lr:.0e}-{time.strftime('%Y%m%d-%H%M%S')}"
    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        config={
            "model": "UnifiedMultimodalDDIModel",
            "modalities": ["HGT_HeteroGraph", "DRKG_TransE", "Chemical_Clinical_Text", "ViT_2D_Image"],
            "hetero_nodes": 2893,
            "drkg_dim": 400,
            "text_dim": 384,
            "vit_dim": drug_vit_embeddings.shape[1],
            "fused_dim": 384,
            "num_classes": 65,
            "batch_size": batch_size,
            "learning_rate": lr,
            "weight_decay": 1e-4,
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE
        }
    )
    
    best_val_f1_this_lr = -1.0
    best_epoch_this_lr = -1
    best_metrics_this_lr = None
    patience_counter = 0
    lr_ckpt_path = WORKING_DIR / f'best_model_lr_{lr:.0e}.pt'
    
    for epoch in range(1, MAX_EPOCHS + 1):
        t_start = time.time()
        model.train()
        running_loss = 0.0
        train_preds = []
        train_targets = []
        
        for idx1, idx2, labels in train_loader:
            idx1, idx2, labels = idx1.to(device), idx2.to(device), labels.to(device)
            optimizer.zero_grad()
            
            logits = model(idx1, idx2, hetero_data=hetero_data, drkg_embs=drkg_embs_dev, text_embs=text_embs_dev, vit_embs=vit_embs_dev)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * len(labels)
            train_preds.extend(torch.argmax(logits, dim=-1).detach().cpu().numpy())
            train_targets.extend(labels.detach().cpu().numpy())
            
        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = accuracy_score(train_targets, train_preds)
        epoch_train_f1 = f1_score(train_targets, train_preds, average='macro', zero_division=0)
        
        # Đánh giá trên tập Validation
        val_m = evaluate_model(model, val_loader, hetero_data, drkg_embs_dev, text_embs_dev, vit_embs_dev, device, num_classes=65)
        
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_m['macro_f1'])
        t_epoch = time.time() - t_start
        
        # Log realtime lên Weights & Biases
        wandb.log({
            "epoch": epoch,
            "train/loss": epoch_train_loss,
            "train/acc": epoch_train_acc,
            "train/macro_f1": epoch_train_f1,
            "val/acc": val_m['acc'],
            "val/macro_f1": val_m['macro_f1'],
            "val/macro_precision": val_m['macro_precision'],
            "val/macro_recall": val_m['macro_recall'],
            "val/micro_auc": val_m['micro_auc'],
            "val/micro_aupr": val_m['micro_aupr'],
            "learning_rate": current_lr
        })
        
        if epoch % 10 == 0 or epoch == 1 or val_m['macro_f1'] > best_val_f1_this_lr:
            print(f"  • Epoch [{epoch:03d}/{MAX_EPOCHS:03d}] ({t_epoch:.1f}s) | "
                  f"Train Loss: {epoch_train_loss:.4f} F1: {epoch_train_f1:.4f} | "
                  f"Val Acc: {val_m['acc']*100:.1f}% F1: {val_m['macro_f1']:.4f} "
                  f"Prec: {val_m['macro_precision']:.4f} Rec: {val_m['macro_recall']:.4f} Micro-AUC: {val_m['micro_auc']:.4f} Micro-AUPR: {val_m['micro_aupr']:.4f}")
        
        # Kiểm tra cải thiện của mức LR này
        if val_m['macro_f1'] > best_val_f1_this_lr:
            best_val_f1_this_lr = val_m['macro_f1']
            best_epoch_this_lr = epoch
            best_metrics_this_lr = val_m
            patience_counter = 0
            torch.save(model.state_dict(), lr_ckpt_path)
            
            # Cập nhật Best Model toàn cục (Global Best Model)
            if best_val_f1_this_lr > global_best_val_f1:
                global_best_val_f1 = best_val_f1_this_lr
                global_best_epoch = epoch
                global_best_lr = lr
                torch.save(model.state_dict(), best_checkpoint_path)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  ✋ [LR {lr:.0e}] Kích hoạt Early Stopping tại Epoch {epoch}! Best Epoch {best_epoch_this_lr} (Val Macro-F1: {best_val_f1_this_lr:.4f})")
                break
                
    # Lưu kết quả so sánh LR này
    lr_experiment_records.append({
        'Learning Rate': f"{lr:.0e}",
        'Best Epoch': best_epoch_this_lr,
        'Val Accuracy (%)': f"{best_metrics_this_lr['acc']*100:.2f}%",
        'Val Macro F1': f"{best_metrics_this_lr['macro_f1']:.4f}",
        'Val Macro Prec': f"{best_metrics_this_lr['macro_precision']:.4f}",
        'Val Macro Rec': f"{best_metrics_this_lr['macro_recall']:.4f}",
        'Val Micro AUC': f"{best_metrics_this_lr['micro_auc']:.4f}",
        'Val Micro AUPR': f"{best_metrics_this_lr['micro_aupr']:.4f}"
    })
    
    wandb.finish()

# ==============================================================================
# TỔNG HỢP SO SÁNH CÁC LEARNING RATE VÀ NẠP CHECKPOINT TỐI ƯU NHẤT
# ==============================================================================
df_lr_summary = pd.DataFrame(lr_experiment_records)
print("\n" + "=" * 90)
print("📊 BẢNG SO SÁNH HIỆU NĂNG GIỮA CÁC LEARNING RATE TRÊN VALIDATION SET")
print("=" * 90)
print(df_lr_summary.to_string(index=False))
print("=" * 90)
print(f"🏆 LEARNING RATE TỐI ƯU NHẤT: {global_best_lr:.0e} (Val Macro-F1 = {global_best_val_f1:.4f} tại Epoch {global_best_epoch})")
print(f"✓ Đã lưu checkpoint tối ưu nhất tại: {best_checkpoint_path}")

# Khôi phục model với cấu hình tốt nhất toàn cục để sẵn sàng đánh giá S0, S1, S2
model = UnifiedMultimodalDDIModel(
    hetero_metadata=hetero_data.metadata(),
    drkg_dim=400,
    text_dim=384,
    vit_dim=drug_vit_embeddings.shape[1],
    proj_dim=256,
    fused_dim=384,
    num_classes=num_classes
).to(device)
model.load_state_dict(torch.load(best_checkpoint_path, weights_only=True))
print(f"✓ Đã nạp thành công Global Best Model từ: {best_checkpoint_path}")


🚀 BẮT ĐẦU THỰC NGHIỆM 3 MỨC LEARNING RATE VỚI TỐI ĐA 100 EPOCHS
• Danh sách Learning Rate: [0.001, 0.0005, 0.0001]
• Early Stopping Patience: 12 epochs

---------------------------------------------------------------------------
▶ [1/3] THỬ NGHIỆM VỚI LEARNING RATE: 1e-03 (Tối đa 100 epochs)
---------------------------------------------------------------------------


epoch,▁▂▃▄▅▆▇█
learning_rate,▁▁▁▁▁▁▁▁
train/acc,▁▃▄▅▆▇██
train/loss,█▄▃▂▂▁▁▁
train/macro_f1,▁▃▄▅▆███
val/acc,▁▂▃▅▇▇██
val/macro_f1,▁▄▄▅▇███
val/macro_precision,▁▃▄▅▇███
val/macro_recall,▁▅▇▇████
val/micro_auc,▁▅▆▇▇███
+1,...


  • Epoch [001/100] (3.1s) | Train Loss: 3.2861 F1: 0.0560 | Val Acc: 12.8% F1: 0.1440 Prec: 0.1602 Rec: 0.4581 Micro-AUC: 0.8612 Micro-AUPR: 0.1154
  • Epoch [002/100] (3.2s) | Train Loss: 1.6607 F1: 0.1817 | Val Acc: 24.8% F1: 0.2975 Prec: 0.2629 Rec: 0.7053 Micro-AUC: 0.9242 Micro-AUPR: 0.2307
  • Epoch [003/100] (3.2s) | Train Loss: 1.0072 F1: 0.2875 | Val Acc: 32.6% F1: 0.3080 Prec: 0.2583 Rec: 0.7711 Micro-AUC: 0.9369 Micro-AUPR: 0.3193
  • Epoch [004/100] (3.2s) | Train Loss: 0.7696 F1: 0.3399 | Val Acc: 41.3% F1: 0.3839 Prec: 0.3118 Rec: 0.8107 Micro-AUC: 0.9604 Micro-AUPR: 0.4199
  • Epoch [005/100] (3.2s) | Train Loss: 0.5329 F1: 0.4067 | Val Acc: 56.2% F1: 0.4717 Prec: 0.3889 Rec: 0.8251 Micro-AUC: 0.9701 Micro-AUPR: 0.5354
  • Epoch [006/100] (3.2s) | Train Loss: 0.3926 F1: 0.5003 | Val Acc: 56.4% F1: 0.4891 Prec: 0.4037 Rec: 0.8266 Micro-AUC: 0.9754 Micro-AUPR: 0.5588
  • Epoch [007/100] (3.1s) | Train Loss: 0.3398 F1: 0.5151 | Val Acc: 62.7% F1: 0.4905 Prec: 0.3963 Rec: 0

epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
learning_rate,███████████████▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
train/acc,▁▂▃▄▅▆▆▆▆▆▆▆▇▇▇▇▇▇█▇████████████████████
train/loss,█▅▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/macro_f1,▁▂▃▄▅▅▅▅▅▅▆▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█████████████
val/acc,▁▂▃▄▅▆▆▆▆▆▆▆▇▇▇▇█▇▇█████████████████████
val/macro_f1,▁▃▄▅▅▅▅▅▅▅▅▆▆▇▆▆▇▇▇▇▇▇▇▇▇███████████████
val/macro_precision,▁▂▂▄▄▅▅▅▄▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█▇████████████
val/macro_recall,▁▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇███████████████
val/micro_auc,▁▅▆▇▇█▇▇▇███████████████████████████████
+1,...



---------------------------------------------------------------------------
▶ [2/3] THỬ NGHIỆM VỚI LEARNING RATE: 5e-04 (Tối đa 100 epochs)
---------------------------------------------------------------------------


  • Epoch [001/100] (3.2s) | Train Loss: 3.5809 F1: 0.0412 | Val Acc: 10.5% F1: 0.1307 Prec: 0.1530 Rec: 0.4607 Micro-AUC: 0.8054 Micro-AUPR: 0.0785
  • Epoch [002/100] (3.3s) | Train Loss: 1.9888 F1: 0.1438 | Val Acc: 22.4% F1: 0.2757 Prec: 0.2505 Rec: 0.7094 Micro-AUC: 0.9023 Micro-AUPR: 0.1831
  • Epoch [004/100] (3.2s) | Train Loss: 0.8653 F1: 0.3375 | Val Acc: 37.3% F1: 0.3548 Prec: 0.2943 Rec: 0.7836 Micro-AUC: 0.9536 Micro-AUPR: 0.3759
  • Epoch [005/100] (3.3s) | Train Loss: 0.6231 F1: 0.3921 | Val Acc: 51.9% F1: 0.4245 Prec: 0.3437 Rec: 0.8017 Micro-AUC: 0.9630 Micro-AUPR: 0.4899
  • Epoch [006/100] (3.3s) | Train Loss: 0.4724 F1: 0.4563 | Val Acc: 58.8% F1: 0.4633 Prec: 0.3765 Rec: 0.8124 Micro-AUC: 0.9733 Micro-AUPR: 0.5721
  • Epoch [007/100] (3.2s) | Train Loss: 0.3639 F1: 0.5129 | Val Acc: 65.1% F1: 0.5218 Prec: 0.4274 Rec: 0.8384 Micro-AUC: 0.9759 Micro-AUPR: 0.6410
  • Epoch [008/100] (3.3s) | Train Loss: 0.3237 F1: 0.5545 | Val Acc: 64.2% F1: 0.5397 Prec: 0.4536 Rec: 0

epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
learning_rate,█████████████▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▃▃▂▂▂▂▂▁▁▁▁▁
train/acc,▁▂▃▄▅▆▆▇▇▇▇▇▇▇▇▇▇███████████████████████
train/loss,█▅▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/macro_f1,▁▂▃▃▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇███▇█████████████
val/acc,▁▂▂▃▅▆▆▆▇▇▇▇▇▇▇▇████████████████████████
val/macro_f1,▁▃▂▃▄▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇█▇█████▇████████████
val/macro_precision,▁▂▂▃▃▄▅▅▆▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇███▇▇▇██████████
val/macro_recall,▁▅▆▆▆▇█▇▇▇▇▇▇█▇▇▇▇█▇████▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
val/micro_auc,▁▅▅▆▇▇▇▇████████████████████████████████
+1,...



---------------------------------------------------------------------------
▶ [3/3] THỬ NGHIỆM VỚI LEARNING RATE: 1e-04 (Tối đa 100 epochs)
---------------------------------------------------------------------------


  • Epoch [001/100] (3.3s) | Train Loss: 4.1090 F1: 0.0109 | Val Acc: 6.7% F1: 0.0944 Prec: 0.1131 Rec: 0.2723 Micro-AUC: 0.7259 Micro-AUPR: 0.0454
  • Epoch [002/100] (3.3s) | Train Loss: 3.5377 F1: 0.0494 | Val Acc: 12.4% F1: 0.1384 Prec: 0.1502 Rec: 0.4495 Micro-AUC: 0.7909 Micro-AUPR: 0.0751
  • Epoch [003/100] (3.3s) | Train Loss: 2.8107 F1: 0.0983 | Val Acc: 14.4% F1: 0.1438 Prec: 0.1461 Rec: 0.5313 Micro-AUC: 0.8146 Micro-AUPR: 0.0956
  • Epoch [004/100] (3.2s) | Train Loss: 2.2283 F1: 0.1493 | Val Acc: 18.2% F1: 0.2020 Prec: 0.1890 Rec: 0.6306 Micro-AUC: 0.8543 Micro-AUPR: 0.1269
  • Epoch [005/100] (3.2s) | Train Loss: 1.8125 F1: 0.1825 | Val Acc: 20.0% F1: 0.2363 Prec: 0.2056 Rec: 0.6776 Micro-AUC: 0.8746 Micro-AUPR: 0.1554
  • Epoch [006/100] (3.2s) | Train Loss: 1.5086 F1: 0.2194 | Val Acc: 23.0% F1: 0.2627 Prec: 0.2235 Rec: 0.7119 Micro-AUC: 0.8893 Micro-AUPR: 0.1849
  • Epoch [007/100] (3.2s) | Train Loss: 1.2840 F1: 0.2601 | Val Acc: 24.0% F1: 0.2767 Prec: 0.2477 Rec: 0.

epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇██
learning_rate,█████████████████▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▂▁
train/acc,▁▁▂▃▃▄▅▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇██████████████████
train/loss,█▇▆▅▄▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/macro_f1,▁▁▂▃▃▃▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇█████████████
val/acc,▁▂▂▃▃▄▅▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇██████████████████
val/macro_f1,▁▁▁▂▂▃▃▄▄▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████████
val/macro_precision,▁▁▂▂▃▃▃▄▄▄▅▅▅▆▆▆▆▆▆▆▇▆▆▇▆▇▇▇▇▇▇▇▇▇██████
val/macro_recall,▁▂▅▆▆▇▆▆▇▇▇▇▇▇▇█▇▇▇▇▇▇▇▇▇██▇████████████
val/micro_auc,▁▃▄▅▆▆▇▇▇▇██████████████████████████████
+1,...



📊 BẢNG SO SÁNH HIỆU NĂNG GIỮA CÁC LEARNING RATE TRÊN VALIDATION SET
Learning Rate  Best Epoch Val Accuracy (%) Val Macro F1 Val Macro Prec Val Macro Rec Val Micro AUC Val Micro AUPR
        1e-03          51           87.32%       0.8153         0.7484        0.9310        0.9978         0.9225
        5e-04          43           86.49%       0.7718         0.7099        0.8819        0.9968         0.9163
        1e-04          67           82.98%       0.7594         0.6799        0.9243        0.9937         0.8748
🏆 LEARNING RATE TỐI ƯU NHẤT: 1e-03 (Val Macro-F1 = 0.8153 tại Epoch 51)
✓ Đã lưu checkpoint tối ưu nhất tại: /kaggle/working/best_unified_multimodal_model.pt
✓ Đã nạp thành công Global Best Model từ: /kaggle/working/best_unified_multimodal_model.pt


## 10. Đánh Giá Đối Chiếu Hiệu Năng Trên 3 Kịch Bản ($S_0, S_1, S_2$)
Đánh giá mô hình trên cả 3 tập kiểm thử theo chuẩn khoa học:
- **$S_0$ (Transductive):** Cả 2 thuốc đều đã biết trong tập huấn luyện.
- **$S_1$ (Semi-inductive):** 1 thuốc đã biết, 1 thuốc mới hoàn toàn (Cold-start 1 phía).
- **$S_2$ (Fully-inductive / Zero-shot):** Cả 2 thuốc đều mới hoàn toàn (Chưa từng thấy lúc train).


In [21]:
print("=== ĐÁNH GIÁ MÔ HÌNH HỢP NHẤT TRÊN 3 TẬP KIỂM THỬ S0, S1, S2 ===")
s0_m = evaluate_model(model, test_s0_loader, hetero_data, drkg_embs_dev, text_embs_dev, vit_embs_dev, device, num_classes=65)
s1_m = evaluate_model(model, test_s1_loader, hetero_data, drkg_embs_dev, text_embs_dev, vit_embs_dev, device, num_classes=65)
s2_m = evaluate_model(model, test_s2_loader, hetero_data, drkg_embs_dev, text_embs_dev, vit_embs_dev, device, num_classes=65)

summary_records = [
    {
        'Kịch Bản Kiểm Thử': 'Test S0 (Transductive)',
        'Bản Chất Cặp Thuốc': 'Cả 2 thuốc đã biết',
        'Số Mẫu': f"{len(test_s0_df):,}",
        'Accuracy (%)': f"{s0_m['acc']*100:.2f}%",
        'Macro Precision': f"{s0_m['macro_precision']:.4f}",
        'Macro Recall': f"{s0_m['macro_recall']:.4f}",
        'Macro F1': f"{s0_m['macro_f1']:.4f}",
        'Micro AUC': f"{s0_m['micro_auc']:.4f}",
        'Micro AUPR': f"{s0_m['micro_aupr']:.4f}"
    },
    {
        'Kịch Bản Kiểm Thử': 'Test S1 (Semi-Inductive)',
        'Bản Chất Cặp Thuốc': '1 thuốc đã biết, 1 thuốc mới',
        'Số Mẫu': f"{len(test_s1_df):,}",
        'Accuracy (%)': f"{s1_m['acc']*100:.2f}%",
        'Macro Precision': f"{s1_m['macro_precision']:.4f}",
        'Macro Recall': f"{s1_m['macro_recall']:.4f}",
        'Macro F1': f"{s1_m['macro_f1']:.4f}",
        'Micro AUC': f"{s1_m['micro_auc']:.4f}",
        'Micro AUPR': f"{s1_m['micro_aupr']:.4f}"
    },
    {
        'Kịch Bản Kiểm Thử': 'Test S2 (Fully-Inductive)',
        'Bản Chất Cặp Thuốc': 'Cả 2 thuốc đều mới (Zero-Shot)',
        'Số Mẫu': f"{len(test_s2_df):,}",
        'Accuracy (%)': f"{s2_m['acc']*100:.2f}%",
        'Macro Precision': f"{s2_m['macro_precision']:.4f}",
        'Macro Recall': f"{s2_m['macro_recall']:.4f}",
        'Macro F1': f"{s2_m['macro_f1']:.4f}",
        'Micro AUC': f"{s2_m['micro_auc']:.4f}",
        'Micro AUPR': f"{s2_m['micro_aupr']:.4f}"
    }
]

df_summary = pd.DataFrame(summary_records)
print("\n" + "=" * 90)
print("🏆 BẢNG TỔNG HỢP HIỆU NĂNG MÔ HÌNH HỢP NHẤT MULTIMODAL (HGT + DRKG + TEXT + ViT)")
print("=" * 90)
print(df_summary.to_string(index=False))
print("=" * 90)

results_csv_path = WORKING_DIR / 'unified_multimodal_results_summary.csv'
df_summary.to_csv(results_csv_path, index=False)
print(f"✓ Đã xuất kết quả ra: {results_csv_path}")

wb_table = wandb.Table(
    columns=["Setting", "Scenario", "Samples", "Accuracy", "Macro Prec", "Macro Rec", "Macro F1", "Micro AUC", "Micro AUPR"]
)
for r in summary_records:
    wb_table.add_data(
        r['Kịch Bản Kiểm Thử'], r['Bản Chất Cặp Thuốc'], r['Số Mẫu'],
        r['Accuracy (%)'], r['Macro Precision'], r['Macro Recall'], r['Macro F1'],
        r['Micro AUC'], r['Micro AUPR']
    )

wandb.log({"evaluation/scenarios_summary_table": wb_table})
wandb.finish()
print("🎉 ĐÃ HOÀN THÀNH TOÀN BỘ TIẾN TRÌNH & ĐỒNG BỘ W&B DASHBOARD THÀNH CÔNG!")


=== ĐÁNH GIÁ MÔ HÌNH HỢP NHẤT TRÊN 3 TẬP KIỂM THỬ S0, S1, S2 ===

🏆 BẢNG TỔNG HỢP HIỆU NĂNG MÔ HÌNH HỢP NHẤT MULTIMODAL (HGT + DRKG + TEXT + ViT)
        Kịch Bản Kiểm Thử             Bản Chất Cặp Thuốc Số Mẫu Accuracy (%) Macro Precision Macro Recall Macro F1 Micro AUC Micro AUPR
   Test S0 (Transductive)             Cả 2 thuốc đã biết  3,502       86.55%          0.7568       0.9062   0.8117    0.9972     0.9180
 Test S1 (Semi-Inductive)   1 thuốc đã biết, 1 thuốc mới 11,602       63.18%          0.5365       0.6380   0.5450    0.9785     0.6681
Test S2 (Fully-Inductive) Cả 2 thuốc đều mới (Zero-Shot)  1,382       42.69%          0.3682       0.3365   0.3230    0.9588     0.3963
✓ Đã xuất kết quả ra: /kaggle/working/unified_multimodal_results_summary.csv


Error: You must call wandb.init() before wandb.log()